# Updating & Mutating Vector Stores (Lecture 29)

Vector stores in production are dynamic: knowledge bases undergo continuous additions, updates, and deletions.
This notebook demonstrates:
- Connecting to an existing vector store.
- Programmatically adding new documents (`add_documents`).
- Tracking collection size before and after mutation.
- Verifying updated knowledge retrieval through an LCEL pipeline.

## 1. Setup & Environment Initialization
Initialize ChromaDB and insert initial documents.

In [1]:
import os
import shutil
import tempfile
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

load_dotenv()
temp_dir = tempfile.mkdtemp(prefix="chroma_update_")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

initial_docs = [
    Document(page_content="Supervised learning maps input features to target labels using labeled data.", metadata={"id": "doc_1"}),
    Document(page_content="Unsupervised learning discovers hidden patterns and clustering in unlabeled data.", metadata={"id": "doc_2"}),
]

vector_store = Chroma.from_documents(
    documents=initial_docs,
    embedding=embeddings,
    persist_directory=temp_dir,
    collection_name="mutation_test_collection"
)

initial_count = vector_store._collection.count()
print(f"Initial collection count: {initial_count}")
assert initial_count == 2, "Expected 2 initial documents" 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Initial collection count: 2


## 2. Ingesting New Knowledge (`add_documents`)
Add Reinforcement Learning documents dynamically.

In [2]:
new_docs = [
    Document(
        page_content="Reinforcement learning trains agents to make sequential decisions by maximizing cumulative rewards through environment interaction.",
        metadata={"id": "doc_3", "topic": "rl"}
    ),
]

added_ids = vector_store.add_documents(new_docs)
updated_count = vector_store._collection.count()

print(f"Added Document IDs: {added_ids}")
print(f"Updated collection count: {updated_count}")

assert updated_count == initial_count + len(new_docs), "Count must reflect added documents"
assert len(added_ids) == len(new_docs), "Returned IDs must match inserted documents" 

Added Document IDs: ['2299cc56-0172-4b94-a4b6-1183cc17ecfd']
Updated collection count: 3


## 3. Querying the Updated Knowledge Base
Verify the newly inserted knowledge is retrievable via LCEL.

In [3]:
retriever = vector_store.as_retriever(search_kwargs={"k": 1})
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)

prompt = ChatPromptTemplate.from_template("""Answer the question based only on the context below:
Context: {context}
Question: {question}
Answer:""")

rag_chain = (
    {"context": retriever | (lambda docs: "\n".join(d.page_content for d in docs)), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rl_query = "How does reinforcement learning train autonomous agents?"
rl_answer = rag_chain.invoke(rl_query)

print(f"Query: {rl_query}")
print(f"Answer: {rl_answer.strip()}")

assert "reward" in rl_answer.lower() or "agent" in rl_answer.lower(), "Answer must reflect the added RL document"

# Cleanup
shutil.rmtree(temp_dir, ignore_errors=True)
print("Mutation test passed and resources cleaned up.")

Query: How does reinforcement learning train autonomous agents?
Answer: Reinforcement learning trains autonomous agents by letting them interact with an environment, observe the resulting states and rewards, and adjust their decision‑making policy so that the cumulative reward over time is maximized.
Mutation test passed and resources cleaned up.
